In [1]:
from pyspark.sql.functions import col, when, to_date, lit, concat, regexp_replace

# 1. Load the "dirty" dataset
df = spark.read.table("cds_dataset")

# 2. Replicate CDS_Master Logic (Cleaning & Categorization)
# Define columns first
spread_cols = ["Spread_Jan_26", "Spread_Feb_26", "Spread_Mar_26", "Spread_Apr_26", "Spread_May_26", "Spread_Jun_26"]

# - Handle "System_Error" and cast ALL spread columns to double (numbers)
# - Handle "nan" by filling nulls with 0
# - Remove duplicates and negative notionals
# - Add "Rating Buckets"
cds_master = df.withColumn("Notional_GBP_M", col("Notional_GBP_M").cast("double")) \
               .withColumn("Spread_Jan_26", when(col("Spread_Jan_26") == "System_Error", 0).otherwise(col("Spread_Jan_26")).cast("double")) \
               .withColumn("Spread_Jan_26", when(col("Spread_Jan_26") == "System_Error", 0).otherwise(col("Spread_Jan_26")).cast("double")) \
               .withColumn("Spread_Feb_26", col("Spread_Feb_26").cast("double")) \
               .withColumn("Spread_Mar_26", col("Spread_Mar_26").cast("double")) \
               .withColumn("Spread_Apr_26", col("Spread_Apr_26").cast("double")) \
               .withColumn("Spread_May_26", col("Spread_May_26").cast("double")) \
               .withColumn("Spread_Jun_26", col("Spread_Jun_26").cast("double")) \
               .fillna(0, subset=spread_cols) \
               .dropDuplicates(["Trade_ID"]) \
               .filter(col("Notional_GBP_M") >= 0) \
               .withColumn("Rating_Buckets", 
                           when(col("Credit_Rating").isin("AAA", "AA", "A", "BBB"), "Investment Grade")
                           .otherwise("High Yield")) \
               .sort("Counterparty_ID")

# 3. Replicate CDS_Transactions
# Just remove the monthly spread columns
cds_transactions = cds_master.drop(*spread_cols) \
    .withColumn("PRIN12_Cat", when(col("Sector").isin("Consumer", "Telecoms"), "Retail Client").otherwise("Professional Client"))

# 4. Replicate CDS_CounterParty
cds_counterparty = cds_master.select("Counterparty_ID").distinct()

# 5. Replicate CDS_Spread (Unpivot Logic)
# Spark uses 'stack' for unpivoting
# unpivot_expr = "stack(6, 'Jan_26', Spread_Jan_26, 'Feb_26', Spread_Feb_26, 'Mar_26', Spread_Mar_26, 'Apr_26', Spread_Apr_26, 'May_26', Spread_May_26, 'Jun_26', Spread_Jun_26) as (Month_Raw, Monthly_Spread)"

cds_spread = cds_master.unpivot(
    ["Trade_ID"],
    ["Spread_Jan_26", "Spread_Feb_26", "Spread_Mar_26", "Spread_Apr_26", "Spread_May_26", "Spread_Jun_26"], 
    "Month_Raw",
    "Monthly_Spread"
    ) \
    .withColumn("Clean_Month", regexp_replace(col("Month_Raw"), "Spread_", "")) \
    .withColumn("Date", to_date(concat(lit("01-"), col("Clean_Month")), "dd-MMM_yy")) \
    .select("Trade_ID", "Date", "Monthly_Spread")

# 6. Save all as optimized Delta Tables for the Semantic Model
cds_master.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("CDS_Master")
cds_transactions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("CDS_Transactions")
cds_counterparty.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("CDS_Counterparty")
cds_spread.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("CDS_Spread")

print("Star Schema Tables created successfully in the Lakehouse!")



StatementMeta(, 6532c59e-76d6-4c5d-8114-0a4ce8d1ac21, 3, Finished, Available, Finished, False)

Star Schema Tables created successfully in the Lakehouse!
